# Pooling, parameters and CNN regularisation — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D022_04_Regularization_and_Visualization.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# Chapter 4: Deep CNNs & Regularization

We have established that CNNs are powerful. But as we make them deeper to solve harder problems, we run into a new enemy: **Overfitting**.

In this chapter, we will learn how to:
1.  **Diagnose Overfitting** using Learning Curves.
2.  **Fight Back** using Regularization (Dropout & Batch Normalization).
3.  **Look Inside** the "black box" to see what our network is actually learning.

---

## 1. The Battle: Overfitting vs. Generalization

When a model memorizes the training data instead of learning patterns, it is **Overfitting**. 

### 1.1 Diagnosing with Learning Curves
The most reliable way to spot this is by comparing **Training Loss** vs **Validation Loss**.

![Overfitting Diagram](https://vitalflux.com/wp-content/uploads/2020/12/overfitting-and-underfitting-wrt-model-error-vs-complexity.png)

*Figure: The Concept of Overfitting - The model fits the noise (dots) rather than the signal (line).*
[Image Source](https://commons.wikimedia.org/)

- **Underfitting**: Both Training and Validation errors are high. The model is too stupid.
- **Optimum**: Validation error is at its lowest. This is the sweet spot.
- **Overfitting**: Training error keeps going down, but Validation error starts going UP. The model is Hallucinating.

Let's create an overfitting model intentionally to see this in action.

In [7]:
! python --version

Python 3.13.9


In [8]:
import numpy as np 
np.__version__

'2.3.5'

In [3]:
! pip show tensorflow

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPooling2D, Dropout, BatchNormalization
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

# 1. Load Data
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
train_images = train_images.reshape((-1, 28, 28, 1)) / 255.0
test_images = test_images.reshape((-1, 28, 28, 1)) / 255.0
train_labels_oh = to_categorical(train_labels)
test_labels_oh = to_categorical(test_labels)

# 2. Build an OVERFITTING Model (Too complex, no regularization)
overfit_model = Sequential([
    Input((28, 28, 1)),
    Conv2D(64, (3, 3), activation='relu'),
    Conv2D(128, (3, 3), activation='relu'), # Too many filters for simple data
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(256, activation='relu'), # Huge Dense layer -> Memorization risk
    Dense(10, activation='softmax')
])

overfit_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train for enough epochs to see the divergence
history = overfit_model.fit(train_images, train_labels_oh, epochs=10, validation_split=0.2, verbose=1)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# TODO: draw a `plt.figure` of `loss`

In [ ]:
# ═══ CHECKPOINT — "The Solution: Regularization" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPooling2D, Dropout, BatchNormalization
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
train_images = train_images.reshape((-1, 28, 28, 1)) / 255.0
test_images = test_images.reshape((-1, 28, 28, 1)) / 255.0
train_labels_oh = to_categorical(train_labels)
test_labels_oh = to_categorical(test_labels)
overfit_model = Sequential([Input((28, 28, 1)), Conv2D(64, (3, 3), activation='relu'), Conv2D(128, (3, 3), activation='relu'), MaxPooling2D((2, 2)), Flatten(), Dense(256, activation='relu'), Dense(10, activation='softmax')])
overfit_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = overfit_model.fit(train_images, train_labels_oh, epochs=1, validation_split=0.2, verbose=1)

print("Checkpoint restored. Ready for: The Solution: Regularization")

## 2. The Solution: Regularization

To fix this, we use two main weapons: **Dropout** and **Batch Normalization**.

### 2.1 Dropout
- **Concept**: Randomly set a fraction of input units to 0 at each update during training time.
- **Analogy**: If a company relies on just one genius, it fails if they are sick. If teams rotate randomly, everyone learns to be robust.
- **Code**: `Dropout(0.5)` means 50% of neurons are turned off each step.

### 2.2 Batch Normalization (BN)
- **Concept**: Normalize the inputs of each layer (mean 0, var 1).
- **Benefit**: Stabilizes learning, allows higher learning rates, acts as slight regularization.

### 2.3 The "Disharmony" (Order Matters)
Research suggests that Dropout can mess up the statistics BN tries to learn.
**Recommended Order**:
$$ Conv2D \rightarrow BatchNormalization \rightarrow Activation \rightarrow Pooling \rightarrow Dropout $$

Let's build a **Robust Model** using this pattern.

In [ ]:
# TODO: compute `robust_model`, `history_robust` from `accuracy`
# Name your result: robust_model, history_robust

In [ ]:
# TODO: draw a `plt.figure` of `loss`

## 3. Interpretation: Looking Inside the Black Box

One of the best things about CNNs is they are visual. We can actually see what they learned.

### 3.1 Visualizing Kernels (Filters)
The first layer filters are usually edge detectors.

In [ ]:
# TODO: compute `filters`, `f`, `ax`
# Name your result: filters, f, ax

### 3.2 Visualizing Feature Maps
Let's pass an image of a **Boot** (or similar) through the network and see what the first Conv layer "sees".

In [ ]:
from tensorflow.keras.models import Model

# 1. Create a model that outputs the FIRST layer's activation
layer_output_model = Model(inputs=robust_model.inputs, outputs=robust_model.layers[0].output)

# 2. Pick a test image
img = test_images[0].reshape(1, 28, 28, 1)

# 3. Get the feature map
feature_maps = layer_output_model.predict(img)

# 4. Plot
plt.figure(figsize=(15, 8))
# Plot original
plt.subplot(1, 7, 1)
plt.title('Original')
plt.imshow(img[0,:,:,0], cmap='gray')

# Plot 6 Feature Maps
for i in range(6):
    plt.subplot(1, 7, i+2)
    plt.imshow(feature_maps[0, :, :, i], cmap='viridis')
    plt.title(f'Map {i}')
plt.show()